# Concordance/Discordance Map: SPM1d vs PCA-SVM

**Project:** SPM1d-PCA-SVM Framework for Quantifying Dynamic Gait Adaptation  
**Author:** Zhang Xining | Shanghai University of Sport

This notebook reproduces **Figure 5** from the paper: a dual-axis butterfly chart visualizing where SPM1d and SVM *agree* vs. *disagree* in their detection of group differences.

- **Left axis (gray):** SPM1d — percentage of stance phase with significant difference
- **Right axis (purple):** PCA-SVM — Contribution Score from feature back-projection
- **Red highlight:** Variables where the two methods most strongly disagree (the Knee Sagittal paradox)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.family'] = 'Arial'

In [ ]:
# ============================================================
# Data — from paper Results (Section 3.1 and 3.2)
# ============================================================

# Selected variables to highlight SPM1d vs SVM discordance
variables = [
    'Pelvis Sagittal (X)',    # SPM: significant (full stance); SVM: low
    'Foot Transverse (Z)',    # SPM: significant; SVM: moderate
    'Knee Frontal (Y)',       # Both: moderate
    'Ankle Sagittal (X)',     # SPM: N.S.; SVM: high (2nd)
    'Knee Transverse (Z)',    # SPM: N.S.; SVM: high (4th), 78.95% accuracy
    'Shank Sagittal (X)',     # SPM: N.S.; SVM: high (3rd)
    'Knee Sagittal (X)',      # *** KEY DISCREPANCY: SPM N.S. but SVM #1 ***
]

# SPM1d: % of stance phase that was significantly different (0 = N.S.)
spm_data = [
    100.0,   # Pelvis Sagittal (0–100%)
    56.7,    # Foot Transverse (41–97.7% ≈ 57%)
    34.7,    # Knee Frontal (65.3–100% ≈ 35%)
    0.0,     # Ankle Sagittal (N.S.)
    0.0,     # Knee Transverse (N.S.)
    0.0,     # Shank Sagittal (N.S.)
    0.0,     # Knee Sagittal (N.S.) ← paradox
]

# SVM: Contribution Score from whole-body feature back-projection
svm_data = [
    0.50,    # Pelvis Sagittal (low, near-proximal joints less discriminative)
    0.86,    # Foot Transverse
    0.81,    # Knee Frontal
    1.66,    # Ankle Sagittal (2nd)
    1.38,    # Knee Transverse (4th)
    1.63,    # Shank Sagittal (3rd)
    1.99,    # Knee Sagittal (1st) ← paradox
]

# Reverse for bottom-to-top display
variables = variables[::-1]
spm_data   = spm_data[::-1]
svm_data   = svm_data[::-1]

In [ ]:
# ============================================================
# Figure: Butterfly Chart
# ============================================================
color_spm       = '#A0A0A0'   # gray — traditional method
color_svm       = '#5B2C8D'   # purple — machine learning
color_highlight = '#D62728'   # red — key discrepancy

fig, ax = plt.subplots(figsize=(13, 6), dpi=200)
y = np.arange(len(variables))

# Left bars: SPM1d (negative → extends left)
ax.barh(y, -np.array(spm_data), color=color_spm,
        height=0.55, label='SPM1d (% significant stance phase)')

# Right bars: SVM (highlight Knee Sagittal)
svm_colors = [
    color_highlight if 'Knee Sagittal' in v else color_svm
    for v in variables
]
ax.barh(y, svm_data, color=svm_colors,
        height=0.55, label='SVM Contribution Score')

# Central axis
ax.axvline(0, color='black', linewidth=1.2)

# Y-axis labels
ax.set_yticks(y)
ax.set_yticklabels(variables, fontsize=11)

# X-axis: symmetric labels
ax.set_xticks([-100, -75, -50, -25, 0, 0.5, 1.0, 1.5, 2.0])
ax.set_xticklabels(['100%','75%','50%','25%','0','0.5','1.0','1.5','2.0'], fontsize=9)

# Axis labels
ax.text(-55, -0.9, '← SPM1d Significant Stance Phase (%)', ha='center',
        fontsize=10, color=color_spm, style='italic')
ax.text(1.0, -0.9, 'SVM Contribution Score →', ha='center',
        fontsize=10, color=color_svm, style='italic')

# Annotation for key discrepancy
knee_sag_idx = variables.index('Knee Sagittal (X)')
ax.annotate(
    'KEY DISCREPANCY\nSPM: N.S.  |  SVM: #1 (1.99)',
    xy=(1.99, knee_sag_idx),
    xytext=(1.5, knee_sag_idx + 1.2),
    fontsize=9, color=color_highlight, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=color_highlight, lw=1.5)
)

# Region labels
ax.text(-85, len(variables)-0.3, 'CONCORDANT\n(Both detect)',
        fontsize=8, color='#555555', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F0F0F0', alpha=0.8))
ax.text(1.7, 1.5, 'DISCORDANT\n(SVM detects,\nSPM misses)',
        fontsize=8, color=color_highlight, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF0F0', alpha=0.8))

# Title & legend
ax.set_title(
    'Concordance & Discordance Map: SPM1d vs PCA-SVM\n'
    'Validating the Magnitude-Structure Complementarity Framework',
    fontsize=13, fontweight='bold', pad=12
)

legend_patches = [
    mpatches.Patch(color=color_spm, label='SPM1d — explicit magnitude differences'),
    mpatches.Patch(color=color_svm, label='SVM — latent structural discriminability'),
    mpatches.Patch(color=color_highlight, label='Key discrepancy (N.S. in SPM, #1 in SVM)'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9,
          framealpha=0.9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('concordance_discordance_map.png', dpi=250, bbox_inches='tight')
plt.show()
print('Saved: concordance_discordance_map.png')

## Interpretation

The chart reveals three distinct variable categories:

| Category | Variables | Meaning |
|----------|-----------|--------|
| **Concordant** | Pelvis Sagittal, Foot Transverse | Both methods agree: explicit postural shift |
| **Partially concordant** | Knee Frontal | Both detect, different magnitudes |
| **Discordant** ⚠️ | **Knee Sagittal, Shank Sagittal, Ankle Sagittal** | SPM misses, SVM captures latent motor signature |

The **Knee Sagittal paradox** is the central finding: despite being statistically invisible to SPM1d (p > 0.05, across the entire waveform), it is the single most important variable for the SVM classifier (Contribution Score: 1.99, discriminative in mid-stance 16–30%). This is the empirical proof that **statistical non-significance ≠ motor restoration**.